# Motor condition monitoring — train the anomaly model in Google Colab

This notebook trains the **same Isolation Forest** the dashboard uses, with the
**same code**, so a model trained here works in the dashboard on your laptop.

**Before you start:** have `predictive_maintenance_colab.zip` ready (it is in the
`colab` folder of the project). Run the cells from top to bottom.

**Honesty note:** by default this trains on *synthetic* data. That model is only
for learning and software testing. To train a model for your real motor, set
`USE_REAL_DATA = True` in step 3 and upload healthy recordings from `data/raw`.

## 1. Match the laptop's scikit-learn version
A saved model must be opened with the same scikit-learn version that created it. The laptop has 1.7.2. If Colab asks you to restart the runtime after this cell, do so, then continue.

In [ ]:
!pip install -q scikit-learn==1.7.2

## 2. Upload the project code
When the button appears, choose `predictive_maintenance_colab.zip`.

In [ ]:
import os, sys, zipfile

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:          # also works in ordinary Jupyter on the laptop
    IN_COLAB = False

if not os.path.exists("predictive_maintenance/config.py"):
    if IN_COLAB and not os.path.exists("predictive_maintenance_colab.zip"):
        files.upload()
    zipfile.ZipFile("predictive_maintenance_colab.zip").extractall(".")

sys.path.insert(0, os.path.abspath("predictive_maintenance"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

import config
from src.ingestion.data_generator import generate_demo_data, generate_accel_bursts, SCENARIOS
from src.monitoring.engine import get_signals
from src.monitoring.session_manager import training_features
from src.processing.feature_pipeline import build_features, feature_columns
from src.detection.isolation_forest import train_model, save_model, load_model, score_features
from src.detection.evaluation import synthetic_pipeline_check

print("scikit-learn", sklearn.__version__)

## 3. Choose the training data

* `USE_REAL_DATA = False` : 12 synthetic "stable baseline" sessions (15 minutes each).
* `USE_REAL_DATA = True`  : you upload healthy recordings from the laptop's
  `data/raw` folder (the `.csv` and, if present, its `_accel.csv`).

Real and synthetic recordings are never combined: the code refuses.

In [ ]:
USE_REAL_DATA = False
OPERATING_NOTES = "synthetic baseline, for software testing"   # describe speed / load / voltage for real data

if USE_REAL_DATA:
    config.RAW_DIR.mkdir(parents=True, exist_ok=True)
    uploaded = files.upload()                      # pick the session CSV files
    for name, content in uploaded.items():
        (config.RAW_DIR / name).write_bytes(content)
    paths = [p for p in sorted(config.RAW_DIR.glob("*.csv")) if not p.name.endswith("_accel.csv")]
    features, data_source, training_id = training_features(paths)
else:
    tables = []
    for seed in range(100, 112):
        df = generate_demo_data("stable_baseline", duration_s=900, seed=seed)
        accel = generate_accel_bursts("stable_baseline", duration_s=900, seed=seed)
        _, bursts = get_signals(df, accel)          # per-burst vibration RMS, peak, FFT peak
        tables.append(build_features(df, bursts))   # one row per 60-second window
    features = pd.concat(tables, ignore_index=True)
    data_source = config.SOURCE_SYNTHETIC
    training_id = "12 synthetic baseline sessions (seeds 100-111)"

print(data_source, "|", len(features), "windows |", training_id)
features.head()

## 4. Look at the features

Each row is one minute of data squeezed into a few numbers. These are what the
model sees. Gas, light and distance are left out on purpose: they describe the
room, not the motor.

In [ ]:
cols = feature_columns(features)
display(features[cols].describe().T[["count", "mean", "std", "min", "max"]].round(4)) if "display" in dir() else print(features[cols].describe().T)
features[cols].hist(figsize=(14, 7), bins=25)
plt.suptitle(f"Training features ({data_source})")
plt.tight_layout()
plt.show()

## 5. Train

Three steps happen inside `train_model`, and the same three are replayed when scoring:

1. **Fill gaps** with the training median.
2. **Scale** every feature to a similar size.
3. **Isolation Forest**: 200 random trees. A window that gets separated from the
   others after only a few splits is "easy to isolate", which means unusual.

The **decision threshold** is the score that 99 % of the training windows stay below.

In [ ]:
pipeline, meta = train_model(features, data_source=data_source, training_id=training_id,
                             operating_notes=OPERATING_NOTES)

print("model version :", meta["model_version"])
print("features      :", meta["features"])
print("windows used  :", meta["n_windows"])
print("threshold     :", round(meta["threshold"], 4))
for w in meta["warnings"]:
    print("WARNING:", w)

train_scores = -pipeline.score_samples(features[meta["features"]].dropna(how="all"))
plt.figure(figsize=(8, 3.5))
plt.hist(train_scores, bins=30)
plt.axvline(meta["threshold"], color="red", linestyle="--", label="decision threshold")
plt.xlabel("anomaly score (higher = more unusual)")
plt.ylabel("training windows")
plt.title(f"Scores of the training data ({data_source})")
plt.legend()
plt.show()

## 6. Try the model on a session it has not seen

Synthetic models can only score synthetic data, so this step uses a generated
scenario. Change `SCENARIO` to any of the names printed below.
The shaded area is where the scenario's change was injected.

In [ ]:
print(list(SCENARIOS))
SCENARIO = "increasing_vibration"

if data_source == config.SOURCE_SYNTHETIC:
    test = generate_demo_data(SCENARIO, duration_s=900, seed=2)
    test_accel = generate_accel_bursts(SCENARIO, duration_s=900, seed=2)
    test_features = build_features(test, get_signals(test, test_accel)[1])
    scores, missing = score_features(pipeline, meta, test_features, data_source)

    plt.figure(figsize=(10, 3.5))
    plt.plot(scores["window_start"], scores["anomaly_score"], marker="o", label="anomaly score")
    plt.axhline(meta["threshold"], color="red", linestyle="--", label="decision threshold")
    labelled = test_features["label_anomaly"].to_numpy()
    if labelled.any():
        plt.axvspan(scores["window_start"][labelled].iloc[0], scores["window_start"].iloc[-1],
                    alpha=0.15, color="orange", label="injected change")
    plt.title(f"SYNTHETIC scenario: {SCENARIO}")
    plt.ylabel("anomaly score")
    plt.legend()
    plt.show()
    print(scores[["window_start", "anomaly_score", "anomaly_prediction"]].round(3).to_string(index=False))
else:
    print("Real-data model: test it in the dashboard on a real session that was not used for training.")

## 7. Compare the three detectors (synthetic software check)

Trains on 12 synthetic sessions and tests thresholds, the statistical baseline and
the Isolation Forest on separately generated scenarios.

* **recall** = share of affected minutes that had an alert
* **false_positive_rate** = share of normal minutes that had an alert anyway

These numbers test the software. They say nothing about your real motor.

In [ ]:
table, _ = synthetic_pipeline_check()
table[["scenario", "detector", "tp", "fp", "fn", "tn", "recall", "false_positive_rate",
       "detection_latency_s"]].round(2)

## 8. Save the model and bring it back to the laptop

Two files are created: the model (`.joblib`) and its record (`.json`).
Put **both** in the project's `models` folder on the laptop. The dashboard will
then list the model in the sidebar under *AI model* (only for data of the same
kind: a synthetic model appears for demo data, a real model for real data).

In [ ]:
version = save_model(pipeline, meta)
reloaded, reloaded_meta = load_model(version)
check = features[meta["features"]].dropna(how="all")
assert np.allclose(pipeline.score_samples(check), reloaded.score_samples(check))
print("Saved and re-loaded correctly:", version)

if IN_COLAB:
    files.download(str(config.MODELS_DIR / f"{version}.joblib"))
    files.download(str(config.MODELS_DIR / f"{version}.json"))
else:
    print("Files are in", config.MODELS_DIR)